In [ ]:
import pandas as pd
import re

import os
import sys

projectRoot = os.path.abspath(os.path.join(os.getcwd(), '..'))
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)
from helpers import initDB

In [ ]:
def draftFixtures():
    xl = pd.read_excel('draftFixtures.xlsx', sheet_name='Fixtures')
    xl.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
    xl = xl.drop(columns=['Unnamed: 1'])
    games = []
    headers = xl.columns[1:]
    for header in headers:
        #account for joint columns
        if '/' in header:
            teams = header.split('/')
        else:
            teams = [header]
        for team in teams:
            teamSeries = xl[header]
            for idx, game in enumerate(teamSeries):
                if pd.isna(game):
                    continue
                gameData = [xl['Date'].iloc[idx], team, idx+1]
                if game == 'BYE' or game == 'Bye':
                    gameData.append('BYE')
                elif game == 'Finals':
                    gameData.append('Finals')
                else:
                    #opponent
                    gameData.append(re.search(r"\(([^)]*)\)", game).group(1)) 
                    #venue
                    if game.split(' (')[0] == 'Home':
                        gameData.append(game[-4:])
                    else:
                        gameData.append(game.split(' (')[0])
                    #times
                    times = game.split(' - ')[1].split(', ')
                    if len(teams) == 1:
                        if ' ' in times[0]:
                            gameData.append(times[0].split(' ')[0])
                        else:
                            gameData.append(times[0])
                    elif len(teams) == 2:
                        if team == teams[0]:
                            gameData.append(times[1].split(' ')[0])
                        elif team == teams[1]:
                            gameData.append(times[0])
                games.append(gameData)
    df = pd.DataFrame(games, columns=['Date', 'Team', 'Round', 'Opponent', 'Venue', 'Time'])
    #typo fixes
    def typoFix(team, round, target, value):
        mask = (df['Team']==team) & (df['Round']==round)
        df.loc[mask, target] = value
    typoFix('A Women', 20, 'Venue', 'MCG1')
    typoFix('A-Res Women', 17, 'Venue', 'MCG1')
    typoFix('D', 1, 'Venue', 'MCG2')
    typoFix('D-Res', 1, 'Venue', 'MCG2')
    df.loc[df['Venue'].isin(['30am', '25pm', '20pm', '05pm']), 'Venue'] = 'MCG1'
    engine = initDB()
    df.to_sql('fixtures', con=engine, if_exists='replace', index=False)
    return df

draftFixtures()